# US-06: Connect Retrieval to GPT-4o via LangChain

This notebook demonstrates the retrieval-to-generation step for the Health & Safety AI RAG pipeline. It shows how the retriever passes relevant chunks into a LangChain prompt, then asks GPT-4o for a grounded answer with source metadata.

The notebook is intentionally designed to work in two modes:
- live mode using the project's ChromaDB + OpenAI setup
- demo mode using a small in-memory retrieval payload when a live vector store is unavailable


In [ ]:
from dotenv import load_dotenv
import os
from pprint import pprint

from src.answer import answer_question
from src.retrieval.retriever import retrieve

load_dotenv(override=True)
api_key = os.environ.get("OPEN_AI_API_KEY") or os.environ.get("OPENAI_API_KEY")
print(f"OpenAI API key configured: {bool(api_key)}")
print("Ready to connect retrieval to GPT-4o.")


## 1. Build a sample retrieval payload

If the vector store is available, the project can fetch live chunks from ChromaDB. If not, we still want a runnable example, so the notebook falls back to a small demo payload that mirrors the metadata the answer chain expects.


In [ ]:
question = "What edge protection do I need when working on a roof?"

demo_results = [{
    "chunk_id": "roof-1",
    "source_file": "working-on-roofs.pdf",
    "page_number": 4,
    "section_heading": "Working at height",
    "text": "Roof work requires edge protection and guardrails. Suitable controls include guardrails, scaffolding, and harness systems when other controls are not practical.",
}]

try:
    live_results = retrieve(question, n_results=4)
    results = live_results
    print(f"Retrieved {len(results)} chunks from ChromaDB")
except Exception as exc:
    results = demo_results
    print(f"Using demo retrieval data because the live vector store is unavailable: {exc}")

pprint(results)


## 2. Connect retrieval to GPT-4o via the answer chain

The project's `answer_question()` helper retrieves the relevant chunks, formats them as prompt context, calls the LLM, and returns the answer plus the sources it used.


In [ ]:
response = answer_question(
    question,
    retriever_fn=lambda q, n_results=None, collection_name=None: results,
)

pprint(response)


## 3. Check the no-results path

If retrieval returns no chunks, the system should fail gracefully and tell the user that no relevant information was found instead of trying to produce an answer.


In [ ]:
empty_response = answer_question(
    "What is the capital of France?",
    retriever_fn=lambda q, n_results=None, collection_name=None: [],
)

pprint(empty_response)


## 4. Check the API error path

The chain should catch LLM issues and return an error payload rather than crashing the application.


In [ ]:
class FailingLLM:
    def invoke(self, payload):
        raise RuntimeError("quota exceeded")

error_response = answer_question(
    question,
    retriever_fn=lambda q, n_results=None, collection_name=None: results,
    llm=FailingLLM(),
)

pprint(error_response)


## 5. Notes for manual verification

The acceptance tests for US-06 are:
- retrieved chunks are passed into the prompt
- the answer includes source metadata (`source_file`, `page_number`, `section_heading`)
- no-results returns a graceful message
- API failures are reported in the response payload instead of crashing

When validating with real PDFs, inspect the output against the original document and page to confirm the sources are correct before using this flow in the app.
